In [1]:
import numpy as np
from cp import run_split_conformal

experiment_name = "resnet18_28_tissuemnist_scratch"
path = f"./outputs/{experiment_name}_outputs.npz"

data = np.load(path)

val_probs = data["val_probs"]
val_labels = data["val_labels"]
test_probs = data["test_probs"]
test_labels = data["test_labels"]

for alpha in [0.1, 0.05]:
    result = run_split_conformal(
        cal_probs=val_probs,
        cal_labels=val_labels,
        test_probs=test_probs,
        test_labels=test_labels,
        alpha=alpha,
    )

    # 🔴 BURASI: qhat ve scores hesaplandıktan sonra
    qhat = result["qhat"]

    val_true_probs = val_probs[np.arange(len(val_labels)), val_labels]
    test_true_probs = test_probs[np.arange(len(test_labels)), test_labels]

    val_scores = 1.0 - val_true_probs
    test_scores = 1.0 - test_true_probs

    print(f"\n=== {experiment_name} | alpha={alpha} ===")
    print(f"qhat: {qhat:.6f}")
    print(result["metrics"])

    print("Calibration coverage wrt qhat:", (val_scores <= qhat).mean())
    print("Test coverage wrt qhat:", (test_scores <= qhat).mean())

    print("Val class counts :", np.bincount(val_labels))
    print("Test class counts:", np.bincount(test_labels))

    print("min prob:", test_probs.min())
    print("max prob:", test_probs.max())


=== resnet18_28_tissuemnist_scratch | alpha=0.1 ===
qhat: 0.972102
{'coverage': 0.8998096446700508, 'avg_set_size': 2.123582910321489, 'singleton_rate': 0.4053934010152284, 'empty_set_rate': 0.0, 'class_wise_coverage': {0: 0.9433564127926146, 1: 0.6972682489923869, 2: 0.8652355396541443, 3: 0.8941390277146751, 4: 0.8453547046601365, 5: 0.8319709355131698, 6: 0.9300062494420142, 7: 0.8812402218745555}}
Calibration coverage wrt qhat: 0.9000846023688663
Test coverage wrt qhat: 0.8998096446700508
Val class counts : [7582 1117  838 2201 1684 1101 5601 3516]
Test class counts: [15165  2233  1677  4402  3369  2202 11201  7031]
min prob: 1.2364548e-18
max prob: 1.0

=== resnet18_28_tissuemnist_scratch | alpha=0.05 ===
qhat: 0.993696
{'coverage': 0.9508671742808799, 'avg_set_size': 2.905456852791878, 'singleton_rate': 0.26694162436548224, 'empty_set_rate': 0.0, 'class_wise_coverage': {0: 0.9773821299043851, 1: 0.8463949843260188, 2: 0.9242695289206917, 3: 0.9502498864152658, 4: 0.9165924606708

In [3]:
import numpy as np
from cp import run_split_conformal, run_class_conditional_conformal

experiment_name = "resnet18_224_tissuemnist_scratch"
path = f"./outputs/{experiment_name}_outputs.npz"

data = np.load(path)

val_probs = data["val_probs"]
val_labels = data["val_labels"]
test_probs = data["test_probs"]
test_labels = data["test_labels"]

for alpha in [0.1, 0.05]:
    print(f"\n==============================")
    print(f"{experiment_name} | alpha={alpha}")
    print(f"==============================")

    # Standard split conformal
    result_std = run_split_conformal(
        cal_probs=val_probs,
        cal_labels=val_labels,
        test_probs=test_probs,
        test_labels=test_labels,
        alpha=alpha,
    )

    print("\n--- Standard split conformal ---")
    print("qhat:", result_std["qhat"])
    print(result_std["metrics"])

    # Class-conditional conformal
    result_cc = run_class_conditional_conformal(
        cal_probs=val_probs,
        cal_labels=val_labels,
        test_probs=test_probs,
        test_labels=test_labels,
        alpha=alpha,
    )

    print("\n--- Class-conditional conformal ---")
    print("qhats:", result_cc["qhats"])
    print(result_cc["metrics"])


resnet18_224_tissuemnist_scratch | alpha=0.1

--- Standard split conformal ---
qhat: 0.921046756207943
{'coverage': 0.8986675126903553, 'avg_set_size': 1.950148054145516, 'singleton_rate': 0.40535109983079526, 'empty_set_rate': 0.0, 'class_wise_coverage': {0: 0.9341246290801187, 1: 0.6377071204657412, 2: 0.8855098389982111, 3: 0.9189004997728305, 4: 0.8272484416740873, 5: 0.8396911898274296, 6: 0.9345594143380055, 7: 0.8910539041388138}}

--- Class-conditional conformal ---
qhats: {0: 0.8622779548168182, 1: 0.9898899113759398, 2: 0.9479419626295567, 3: 0.8924925327301025, 4: 0.9566162079572678, 5: 0.9635353423655033, 6: 0.8595346361398697, 7: 0.9294342249631882}
{'coverage': 0.8997250423011844, 'avg_set_size': 2.20748730964467, 'singleton_rate': 0.32151015228426394, 'empty_set_rate': 0.0, 'class_wise_coverage': {0: 0.9008242664029014, 1: 0.9113300492610837, 2: 0.9242695289206917, 3: 0.8941390277146751, 4: 0.8857227663995251, 5: 0.9091734786557675, 6: 0.8977769841978395, 7: 0.898165268

In [4]:
import numpy as np

TISSUEMNIST_CLASSES = {
    0: "Collecting Duct, Connecting Tubule",
    1: "Distal Convoluted Tubule",
    2: "Glomerular endothelial cells",
    3: "Interstitial endothelial cells",
    4: "Leukocytes",
    5: "Podocytes",
    6: "Proximal Tubule Segments",
    7: "Thick Ascending Limb",
}


def compute_cp_metrics(pred_sets, test_labels, class_names=None):
    """
    pred_sets: shape (n_samples, n_classes), bool
    test_labels: shape (n_samples,)
    """

    test_labels = test_labels.reshape(-1).astype(int)
    set_sizes = pred_sets.sum(axis=1)

    covered = pred_sets[np.arange(len(test_labels)), test_labels]

    metrics = {
        "global_coverage": covered.mean(),
        "mean_set_size": set_sizes.mean(),
        "singleton_rate": np.mean(set_sizes == 1),
        "empty_rate": np.mean(set_sizes == 0),
        "per_class_coverage": {},
        "per_class_mean_set_size": {},
    }

    num_classes = pred_sets.shape[1]

    for c in range(num_classes):
        idx = test_labels == c

        if idx.sum() == 0:
            metrics["per_class_coverage"][c] = np.nan
            metrics["per_class_mean_set_size"][c] = np.nan
            continue

        metrics["per_class_coverage"][c] = covered[idx].mean()
        metrics["per_class_mean_set_size"][c] = set_sizes[idx].mean()

    return metrics

def run_class_conditional_conformal(
    cal_probs,
    cal_labels,
    test_probs,
    test_labels,
    alpha=0.1,
):
    cal_labels = cal_labels.reshape(-1).astype(int)
    test_labels = test_labels.reshape(-1).astype(int)

    num_classes = cal_probs.shape[1]
    qhats = {}

    for c in range(num_classes):
        idx = cal_labels == c
        scores_c = 1 - cal_probs[idx, c]

        n = len(scores_c)
        q_level = np.ceil((n + 1) * (1 - alpha)) / n
        q_level = min(q_level, 1.0)

        qhats[c] = np.quantile(scores_c, q_level, method="higher")

    pred_sets = np.zeros_like(test_probs, dtype=bool)

    for c in range(num_classes):
        threshold = 1 - qhats[c]
        pred_sets[:, c] = test_probs[:, c] >= threshold

    metrics = compute_cp_metrics(
        pred_sets=pred_sets,
        test_labels=test_labels,
        class_names=TISSUEMNIST_CLASSES,
    )

    return {
        "qhats": qhats,
        "prediction_sets": pred_sets,
        "metrics": metrics,
    }

In [5]:
def print_class_conditional_results(result, alpha=0.1, model_name="ResNet-18"):
    qhats = result["qhats"]
    metrics = result["metrics"]

    confidence = int((1 - alpha) * 100)

    print(f"\n--- CLASS-CONDITIONAL THRESHOLDS FOR {confidence}% CONFIDENCE ({model_name}) ---")

    for c, qhat in qhats.items():
        print(
            f"Class {c} Threshold: {qhat:.4f}"
        )

    print("\n============================================================")
    print(f"{model_name} + CLASS-CONDITIONAL CP RESULTS (TissueMNIST)")
    print("============================================================")
    print(f"Global Coverage: {metrics['global_coverage'] * 100:.2f}%")
    print(f"Mean Set Size:   {metrics['mean_set_size']:.2f} options per image")

    print("\n--- Per-Class Coverage Breakdown ---")
    for c, cov in metrics["per_class_coverage"].items():
        name = TISSUEMNIST_CLASSES.get(c, f"Class {c}")
        print(f"Class {c} ({name}): {cov * 100:.2f}%")

    print("\n============================================================")
    print("MEAN SET SIZE PER CLASS (CP UNCERTAINTY ANALYSIS)")
    print("============================================================")
    print(f"Global Ortalama Set Boyutu (Mean Set Size): {metrics['mean_set_size']:.2f}")

    print("\n--- Sınıf Bazında Ortalama Seçenek Sayısı (Zorluk Derecesi) ---")
    for c, mss in metrics["per_class_mean_set_size"].items():
        name = TISSUEMNIST_CLASSES.get(c, f"Class {c}")
        print(f"Class {c} ({name}): Ortalama {mss:.2f} seçenek sunuldu.")

In [6]:
result_cc = run_class_conditional_conformal(
    cal_probs=val_probs,
    cal_labels=val_labels,
    test_probs=test_probs,
    test_labels=test_labels,
    alpha=0.1,
)

print_class_conditional_results(
    result_cc,
    alpha=0.1,
    model_name="ResNet-18"
)


--- CLASS-CONDITIONAL THRESHOLDS FOR 90% CONFIDENCE (ResNet-18) ---
Class 0 Threshold: 0.8623
Class 1 Threshold: 0.9899
Class 2 Threshold: 0.9479
Class 3 Threshold: 0.8925
Class 4 Threshold: 0.9566
Class 5 Threshold: 0.9635
Class 6 Threshold: 0.8595
Class 7 Threshold: 0.9294

ResNet-18 + CLASS-CONDITIONAL CP RESULTS (TissueMNIST)
Global Coverage: 89.97%
Mean Set Size:   2.21 options per image

--- Per-Class Coverage Breakdown ---
Class 0 (Collecting Duct, Connecting Tubule): 90.08%
Class 1 (Distal Convoluted Tubule): 91.13%
Class 2 (Glomerular endothelial cells): 92.43%
Class 3 (Interstitial endothelial cells): 89.41%
Class 4 (Leukocytes): 88.57%
Class 5 (Podocytes): 90.92%
Class 6 (Proximal Tubule Segments): 89.78%
Class 7 (Thick Ascending Limb): 89.82%

MEAN SET SIZE PER CLASS (CP UNCERTAINTY ANALYSIS)
Global Ortalama Set Boyutu (Mean Set Size): 2.21

--- Sınıf Bazında Ortalama Seçenek Sayısı (Zorluk Derecesi) ---
Class 0 (Collecting Duct, Connecting Tubule): Ortalama 2.06 seçenek s

In [7]:
experiment_names = [
    "resnet18_28_tissuemnist_scratch",
    "resnet18_224_tissuemnist_scratch",
]

for experiment_name in experiment_names:
    path = f"./outputs/{experiment_name}_outputs.npz"
    data = np.load(path)

    val_probs = data["val_probs"]
    val_labels = data["val_labels"]
    test_probs = data["test_probs"]
    test_labels = data["test_labels"]

    model_label = "ResNet-18 (28)" if "28" in experiment_name else "ResNet-18 (224)"

    for alpha in [0.1, 0.05]:
        result_cc = run_class_conditional_conformal(
            cal_probs=val_probs,
            cal_labels=val_labels,
            test_probs=test_probs,
            test_labels=test_labels,
            alpha=alpha,
        )

        print_class_conditional_results(
            result_cc,
            alpha=alpha,
            model_name=model_label,
        )


--- CLASS-CONDITIONAL THRESHOLDS FOR 90% CONFIDENCE (ResNet-18 (28)) ---
Class 0 Threshold: 0.9122
Class 1 Threshold: 0.9974
Class 2 Threshold: 0.9918
Class 3 Threshold: 0.9675
Class 4 Threshold: 0.9882
Class 5 Threshold: 0.9931
Class 6 Threshold: 0.9424
Class 7 Threshold: 0.9817

ResNet-18 (28) + CLASS-CONDITIONAL CP RESULTS (TissueMNIST)
Global Coverage: 89.93%
Mean Set Size:   2.41 options per image

--- Per-Class Coverage Breakdown ---
Class 0 (Collecting Duct, Connecting Tubule): 89.98%
Class 1 (Distal Convoluted Tubule): 90.33%
Class 2 (Glomerular endothelial cells): 91.65%
Class 3 (Interstitial endothelial cells): 88.60%
Class 4 (Leukocytes): 89.17%
Class 5 (Podocytes): 90.42%
Class 6 (Proximal Tubule Segments): 90.13%
Class 7 (Thick Ascending Limb): 89.99%

MEAN SET SIZE PER CLASS (CP UNCERTAINTY ANALYSIS)
Global Ortalama Set Boyutu (Mean Set Size): 2.41

--- Sınıf Bazında Ortalama Seçenek Sayısı (Zorluk Derecesi) ---
Class 0 (Collecting Duct, Connecting Tubule): Ortalama 2.07